# From pixels to embeddings: computing TESSERA v2 embeddings on your laptop

In this notebook you take a real Sentinel-2 time series of a small Alpine valley, pass it through a pretrained Earth observation foundation model (**TESSERA v2**) and get back an **embedding** for every 10 m pixel, i.e. a vector of 128 numbers that summarises how this pixel looked and changed over one year. You then explore what these vectors capture and where their limits are.

**Learning objectives.** After this exercise you can

1. explain what a pixel embedding is and how a model like TESSERA derives it from a satellite time series,
2. prepare Sentinel-2 L2A data for a pretrained model (band order, cloud mask, scaling, acquisition dates),
3. visualise embeddings with PCA and explore them with clustering and similarity search,
4. critically judge what these unsupervised results do and do not show.

**What you need.** A web browser and a Google account for Google Colab (recommended), or alternatively a laptop with Anaconda (see Part 0). No installation is needed in Colab. Downloading the data and computing the embeddings takes about 5 to 15 minutes, everything else runs in seconds.

**How to work through this notebook.** Run the cells from top to bottom with **Shift + Enter**. Code cells you are meant to change are marked with **✏️ Your turn**. Intermediate results are saved to disk, so if the kernel crashes or you come back another day, re-running the notebook loads them instead of downloading and computing everything again.

### The big picture

```
Sentinel-2 time series                    TESSERA encoder                   Embedding
(one pixel, one year)                     (pretrained, frozen)              (one pixel)

date 1:  B04 B02 B03 ... B12  ─┐
date 2:  B04 B02 B03 ... B12   │      ┌──────────────────────────┐
date 3:  (cloud, skipped)      ├────▶ │  Transformer over time   │ ────▶  [0.81, -1.20, 0.05, ... ]
 ...                           │      │  + day-of-year encoding  │          128 numbers
date n:  B04 B02 B03 ... B12  ─┘      └──────────────────────────┘
```

The model works **pixel by pixel**. It looks at the sequence of clear observations of one pixel, knows on which day of the year each was taken, and compresses everything into 128 numbers. Neighbouring pixels are not used. Any spatial pattern you will see in the maps below therefore comes from the landscape itself, not from the model looking at the neighbourhood.

## Part 0 · Setup

### Option A: Google Colab (recommended)

Colab runs the notebook on a Google server in your browser. Nothing needs to be installed on your computer.

1. Open the notebook in Colab: go to [colab.research.google.com](https://colab.research.google.com), sign in with a Google account and choose *File → Upload notebook*. (If you got a direct "Open in Colab" link, simply click it.)
2. Optional but faster: *Runtime → Change runtime type → T4 GPU → Save*. The embeddings are then computed on a graphics card. The free GPU is not always available; the CPU also works, it just takes a bit longer.
3. Run the cells from top to bottom with **Shift + Enter**. The first code cell installs a few missing packages (about 1 minute). If Colab shows a warning *"This notebook was not authored by Google"*, click *Run anyway*. If it asks you to *restart the session*, do so and run the notebook again from the top.

**Your files in Colab are temporary.** Everything the notebook saves disappears when the session ends (after closing the tab or about 90 minutes without activity). You have two options:

- Set `SAVE_TO_DRIVE = True` in the next code cell. Colab then asks for access to your Google Drive and saves all results in the folder `tessera_output` there. Data and embeddings survive the session, so a later run loads them instead of downloading and computing again.
- Or download the results at the end with the last cell of this notebook (Part 9).

### Option B: Your own computer (Anaconda)

If you prefer to work locally, follow the separate setup guide. In short, run once in the **Anaconda Prompt**:

```cmd
conda create -n tessera python=3.11 -y
conda activate tessera
python -m pip install "torch>=2.4" numpy matplotlib scikit-learn rasterio pystac-client planetary-computer odc-stac huggingface-hub requests ipykernel jupyterlab umap-learn
python -m ipykernel install --user --name tessera --display-name "Python (tessera)"
```

and every time you want to work on the notebook:

```cmd
conda activate tessera
cd %USERPROFILE%\Downloads
python -m jupyterlab
```

Select the kernel **Python (tessera)** in the top right corner of the notebook. Results are saved in the folder `tessera_output` next to the notebook.

The next cell checks where the notebook is running, installs missing packages and decides where results are saved.

In [ ]:
import os, sys, time, warnings, importlib
from pathlib import Path

SAVE_TO_DRIVE = False   # Colab only: True = keep all results in your Google Drive (asks for permission)

IN_COLAB = "google.colab" in sys.modules

# Silence a harmless Windows warning from the Hugging Face download cache
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Check that all required packages are installed (import name -> pip name)
required = {"torch": "torch", "numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn",
            "rasterio": "rasterio", "pystac_client": "pystac-client", "planetary_computer": "planetary-computer",
            "odc.stac": "odc-stac", "huggingface_hub": "huggingface-hub", "requests": "requests"}
missing = []
for module, pip_name in required.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(pip_name)
if missing:
    # Install whatever is missing into the environment this notebook runs in (Colab, Kaggle or local)
    import subprocess
    print("Installing missing packages, this can take a few minutes:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing, "umap-learn"])
    importlib.invalidate_caches()
    print("Done.")

import numpy as np
import matplotlib.pyplot as plt
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Where results are saved
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/tessera_output")
else:
    OUT = Path.cwd() / "tessera_output"
OUT.mkdir(parents=True, exist_ok=True)

print("Running in   :", "Google Colab" if IN_COLAB else "local Python", f"({sys.executable})")
print("PyTorch      :", torch.__version__, "| computing on:", DEVICE)
print("Results go to:", OUT)
if IN_COLAB and not SAVE_TO_DRIVE:
    print("Note: files in Colab are deleted when the session ends. Download them with Part 9, "
          "or set SAVE_TO_DRIVE = True and run this cell again.")

## Part 1 · Study area and settings

The default study area is a 6 × 7 km section of the **Salzach valley near Hallein and Kuchl** (Salzburg, Austria). It contains the river, meadows, arable land, villages, a motorway, deciduous and coniferous forest and steep slopes, which gives the model plenty of contrasts to work with.

Computation time and memory grow with the number of pixels. As a rule of thumb, 1 million pixels (10 × 10 km) need about 2 to 3 GB of RAM and several minutes of computing time on a laptop.

**✏️ Your turn (optional).** Choose another preset or enter your own bounding box. You can get coordinates by right-clicking a location in Google Maps or with the *Copy coordinate* tool in QGIS. Keep the area below about 10 × 10 km.

In [ ]:
# Study areas as [west, south, east, north] in WGS84 degrees
AOI_PRESETS = {
    "salzach_small": [13.06, 47.60, 13.14, 47.66],   # ~6 × 7 km, about 1–3 min of computing (default)
    "salzach_large": [13.03, 47.58, 13.17, 47.68],   # ~10 × 11 km, about 5–10 min
}
AOI_NAME = "salzach_small"
BBOX = AOI_PRESETS[AOI_NAME]
# Own area: uncomment and edit the next two lines
# AOI_NAME = "my_area"
# BBOX = [13.00, 47.70, 13.06, 47.75]

YEAR = 2024             # one calendar year of observations
DATES_PER_MONTH = 2     # number of clearest acquisition dates kept per month (see Part 3)
MAX_CLOUD = 60          # ignore scenes with more than this % cloud cover (whole scene, not your area)
MODEL = "nano"          # nano (~4 MB) | small (~28 MB) | medium (~84 MB)
BATCH_PIXELS = 512      # pixels processed at once; use 128 if your computer runs out of memory
CRS = "EPSG:32633"      # UTM zone 33N, metric coordinates for Salzburg. Use the UTM zone of your area.

# Rough size estimate so you know what to expect
km_x = (BBOX[2] - BBOX[0]) * 111.32 * np.cos(np.radians((BBOX[1] + BBOX[3]) / 2))
km_y = (BBOX[3] - BBOX[1]) * 110.57
n_pixels = int(km_x * 100 * km_y * 100)
print(f"Study area '{AOI_NAME}': {km_x:.1f} × {km_y:.1f} km ≈ {n_pixels/1e6:.2f} million pixels at 10 m")
if n_pixels > 1.5e6:
    print("⚠ This area is large for a laptop. Expect long run times and high memory use.")

## Part 2 · The model: TESSERA v2

**What TESSERA is.** TESSERA is an Earth observation foundation model developed at the University of Cambridge. It was trained on Sentinel-1 radar and Sentinel-2 optical time series from around the world, **without any land cover labels**. Instead of learning "this is forest", it learned to describe pixels in a way that is useful for many later tasks (self-supervised learning). Its output is a 128-dimensional embedding per pixel and year.

**What happens inside**, per pixel:

1. Only **clear observations** are used. Their number varies from pixel to pixel, so the sequence is resampled to a fixed length (8, 16, 24, … observations).
2. Each observation is the 10 Sentinel-2 bands plus its **day of year**. The day of year matters because it tells the model *when* something happened, e.g. that a meadow was green in May and cut in June.
3. A **Transformer** relates all observations of the year to each other, and an attention-pooling step summarises them into one vector.
4. A final layer produces **128 numbers**, normalised so that each pixel's vector has mean 0 and standard deviation 1.

The 128 dimensions are **Matryoshka-ordered**: like Russian nesting dolls, the first 16, 32 or 64 numbers already form a usable, smaller embedding. The first dimensions carry the most important information.

We use the **Student** version of TESSERA v2. It is a small model *distilled* from the large original (it learned to imitate the large model's outputs), which is why it runs on a laptop. `nano` has about 1 million parameters.

**Important limitation of this exercise.** TESSERA expects Sentinel-1 *and* Sentinel-2. To keep things simple we only provide Sentinel-2, so the radar branch of the model receives an empty input. The embeddings are therefore **not identical** to the official TESSERA products and will be weaker where radar would help, e.g. under persistent clouds.

The next cell downloads the model code directly from the official TESSERA repository on GitHub, and the trained weights from Hugging Face. Both are cached, so this only takes time on the first run. We do not write our own version of the model, so what you run is exactly what the authors published.

In [ ]:
import requests
from huggingface_hub import hf_hub_download

# 1) Model code from the official repository
CODE_DIR = OUT / "tessera_student"
CODE_DIR.mkdir(exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/ucam-eo/tessera/master/tessera_infer_v2/student/"
for filename in ("model.py", "infer.py"):
    target = CODE_DIR / filename
    if not target.exists():
        response = requests.get(BASE_URL + filename, timeout=60)
        response.raise_for_status()
        target.write_bytes(response.content)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from model import load_model
from infer import encode_tile

# 2) Pretrained weights from Hugging Face
HF_REPOS = {
    "nano":   ("geotessera/TESSERA-V-2.0-2B-N", "ckpt/student_nano.pt"),
    "small":  ("geotessera/TESSERA-V-2.0-2B-S", "ckpt/student_small.pt"),
    "medium": ("geotessera/TESSERA-V-2.0-2B-M", "ckpt/student_medium.pt"),
}
repo_id, ckpt_file = HF_REPOS[MODEL]
checkpoint = hf_hub_download(repo_id=repo_id, filename=ckpt_file)
model = load_model(checkpoint, DEVICE)   # also switches the model to inference mode

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded TESSERA v2 student '{MODEL}' with {n_params:,} parameters")

## Part 3 · Sentinel-2 data

### 3.1 Finding scenes

We search the **Microsoft Planetary Computer**, which provides the complete Sentinel-2 L2A archive (atmospherically corrected surface reflectance) as Cloud Optimized GeoTIFFs. "Cloud optimized" means we can download just our small area instead of the full 110 × 110 km tile.

Because TESSERA is a *time series* model, the **number and spread of dates** matters more than having a few perfect scenes. We therefore keep the `DATES_PER_MONTH` clearest dates of every month. Picking only the clearest scenes of the whole year would give you mostly summer images, and an Alpine pixel would end up with two or three observations, which leaves the model little to work with.

Note that the cloud cover in the metadata refers to the whole tile. Your area may still be cloudy on a "clear" day, or clear on a cloudy one. The pixel-wise cloud mask in Part 4 takes care of that.

In [ ]:
from collections import defaultdict
from pystac_client import Client
import planetary_computer

CACHE_FILE = OUT / f"s2_{AOI_NAME}_{YEAR}.npz"   # downloaded data are stored here

if CACHE_FILE.exists():
    print(f"Found saved data in {CACHE_FILE.name}, skipping the search and download.")
else:
    catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=BBOX,
        datetime=f"{YEAR}-01-01/{YEAR}-12-31",
        query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    )
    items = list(search.items())
    if not items:
        raise RuntimeError("No scenes found. Check BBOX, YEAR and MAX_CLOUD.")

    # Several tiles can cover the same area on the same day. Group them by date.
    items_by_date = defaultdict(list)
    for item in items:
        items_by_date[item.datetime.date()].append(item)
    cloud_by_date = {d: np.mean([i.properties["eo:cloud_cover"] for i in its])
                     for d, its in items_by_date.items()}

    # Keep the clearest dates of every month
    selected_dates = []
    for month in range(1, 13):
        dates_in_month = [d for d in cloud_by_date if d.month == month]
        selected_dates += sorted(dates_in_month, key=cloud_by_date.get)[:DATES_PER_MONTH]
    selected_dates = sorted(selected_dates)
    selected_items = [item for d in selected_dates for item in items_by_date[d]]

    print(f"{len(items)} scenes on {len(items_by_date)} dates found, "
          f"{len(selected_dates)} dates selected ({len(selected_items)} scenes):")
    for d in selected_dates:
        print(f"  {d}   scene cloud cover {cloud_by_date[d]:5.1f} %")

### 3.2 Downloading

Now the actual pixels are downloaded: 10 spectral bands plus the **scene classification layer (SCL)**, which we use as cloud mask. All bands are resampled to a common 10 m grid in UTM coordinates. Bands with 20 m resolution (B05–B07, B8A, B11, B12, SCL) are simply split into 2 × 2 pixels (nearest neighbour), which does not add any detail.

Scenes from the same day are merged (`groupby="solar_day"`), so every time step is one acquisition date.

⏱ This is the slowest part of the notebook (typically 2 to 10 minutes). The result is saved, so you only do this once per study area.

In [ ]:
# Band order expected by TESSERA. Do not change it, the model was trained with exactly this order.
BANDS = ["B04", "B02", "B03", "B08", "B8A", "B05", "B06", "B07", "B11", "B12"]

if not CACHE_FILE.exists():
    from odc.stac import load
    t0 = time.time()
    signed_items = [planetary_computer.sign(item) for item in selected_items]   # adds temporary access tokens
    ds = load(
        signed_items,
        bands=BANDS + ["SCL"],
        bbox=BBOX,
        crs=CRS,
        resolution=10,
        resampling="nearest",
        groupby="solar_day",
        fail_on_error=False,
    )
    raw = np.stack([ds[b].values for b in BANDS], axis=-1)    # (time, rows, cols, bands), uint16
    scl = ds["SCL"].values.astype(np.uint8)                  # (time, rows, cols)
    dates = ds.time.values.astype("datetime64[D]")
    geobox = ds.odc.geobox
    np.savez(CACHE_FILE, raw=raw, scl=scl, dates=dates.astype(str),
             crs=geobox.crs.to_wkt(), transform=np.array(tuple(geobox.transform)[:6]))
    del ds
    print(f"Downloaded in {(time.time() - t0)/60:.1f} min and saved to {CACHE_FILE.name}")

# Load (from the file we just wrote, or from an earlier session)
cache = np.load(CACHE_FILE)
raw, scl = cache["raw"], cache["scl"]
dates = cache["dates"].astype("datetime64[D]")
CRS_WKT, TRANSFORM = str(cache["crs"]), tuple(cache["transform"])
T, H, W, _ = raw.shape
print(f"Data cube: {T} dates × {H} rows × {W} columns × {len(BANDS)} bands")

## Part 4 · Preparing the input

Three steps turn the downloaded values into the input TESSERA expects. Each of them repeats what was done when the model was trained. If the input differs from the training data, the model still produces numbers, but they become less meaningful. This is one of the most common silent errors when using pretrained models.

**1. Removing the reflectance offset.** Sentinel-2 L2A values are reflectance × 10 000 (a value of 3000 means 30 % reflectance). Since the processing baseline 04.00 (25 January 2022), ESA adds **+1000** to every value so that negative reflectances can be stored. The Planetary Computer delivers the values with this offset, so we subtract it, exactly as the official TESSERA preprocessing does. Without this step, every input would be shifted, and spectral indices such as NDVI would be wrong too.

**2. Cloud mask.** The SCL classifies each pixel on each date. Following the official TESSERA preprocessing, we discard *no data* (0), *saturated/defective* (1), *dark areas/topographic shadow* (2), *cloud shadow* (3) and *clouds* (8, 9). Vegetation, bare soil, water, thin cirrus and snow are kept. The SCL is not perfect, so some cloud edges and shadows will slip through.

**3. Day of year.** For every date we compute the day of year (1–365), which the model needs to place each observation within the seasonal cycle.

The model standardises the band values itself, so we pass reflectance × 10 000, **not** values scaled to 0–1.

In [ ]:
# 1) Remove the +1000 offset for acquisitions after the baseline change (only where a value exists)
OFFSET, OFFSET_START = 1000, np.datetime64("2022-01-25")
has_data = (raw > 0).all(axis=-1)                      # 0 means "no data" in the raw files
s2 = raw.astype(np.float32)
for t in np.where(dates > OFFSET_START)[0]:
    s2[t] = np.where(s2[t] >= OFFSET, s2[t] - OFFSET, s2[t])

# 2) Cloud mask from the scene classification layer: 1 = clear, 0 = do not use
SCL_INVALID = [0, 1, 2, 3, 8, 9]
clear = has_data & ~np.isin(scl, SCL_INVALID)
masks = clear.astype(np.uint8)

# 3) Day of year for each date (leap day 366 is mapped to 365)
doys = (dates - dates.astype("datetime64[Y]")).astype(int) + 1
doys = np.minimum(doys, 365).astype(np.int32)

valid_counts = masks.sum(axis=0)                       # clear observations per pixel
print(f"Clear observations per pixel: median {np.median(valid_counts):.0f}, "
      f"min {valid_counts.min()}, max {valid_counts.max()}")
print(f"Pixels without any clear observation: {(valid_counts == 0).sum()} of {valid_counts.size}")

### 4.1 Look at your data before you model it

Before trusting any model output, check what goes in. The figure shows (a) the clearest summer image in true color (used as reference map in the rest of the notebook), (b) how much of the area was cloud-free on each date, and (c) how many clear observations each pixel has. Pixels with only a few observations get less reliable embeddings.

In [ ]:
def true_color(t, gamma=0.8):
    "Simple true-color image of date index t (reflectance 0–0.25 stretched to 0–1)."
    rgb = s2[t][..., [BANDS.index("B04"), BANDS.index("B03"), BANDS.index("B02")]] / 2500
    return np.clip(rgb, 0, 1) ** gamma

clear_fraction = masks.mean(axis=(1, 2))
# Reference image for all later maps: the clearest date between May and September (green vegetation, no snow)
month = dates.astype("datetime64[M]").astype(int) % 12 + 1
summer = (month >= 5) & (month <= 9)
t_best = int(np.argmax(np.where(summer, clear_fraction, -1))) if summer.any() else int(np.argmax(clear_fraction))
rgb_best = true_color(t_best)

fig = plt.figure(figsize=(14, 4.5))
ax1 = fig.add_subplot(1, 3, 1)
ax1.imshow(rgb_best)
ax1.set_title(f"(a) True color, {dates[t_best]}")
ax2 = fig.add_subplot(1, 3, 2)
ax2.bar(doys, clear_fraction * 100, width=6, color="tab:blue")
ax2.set(xlabel="Day of year", ylabel="Cloud-free area (%)", title="(b) Clear pixels per date", xlim=(0, 366), ylim=(0, 100))
ax3 = fig.add_subplot(1, 3, 3)
im = ax3.imshow(valid_counts, cmap="viridis")
ax3.set_title("(c) Clear observations per pixel")
fig.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
for ax in (ax1, ax3):
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### 4.2 What the model sees for a single pixel

This is the actual input for one pixel: its NDVI on every clear date. (The model receives all 10 bands, NDVI is just easier to read.) Compare a forest pixel with a meadow or a field. Meadows typically show drops after each cut, fields a sharp rise and fall, forests a smoother curve.

**✏️ Your turn.** Find two pixels with different land cover in image (a) above. The axes of the image below show row and column numbers. Enter them in `PIXELS` and re-run the cell.

In [ ]:
PIXELS = {"pixel A": (H // 3, W // 3), "pixel B": (2 * H // 3, 2 * W // 3)}   # name: (row, col)

nir, red = s2[..., BANDS.index("B08")], s2[..., BANDS.index("B04")]
with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = (nir - red) / (nir + red)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.6]})
axes[0].imshow(rgb_best)
axes[0].set_title("Location of the pixels (row/col)")
for (name, (r, c)), color in zip(PIXELS.items(), ["red", "orange", "cyan", "magenta"]):
    ok = masks[:, r, c] == 1
    axes[0].scatter(c, r, s=80, marker="x", c=color, linewidths=2.5)
    axes[1].plot(doys[ok], ndvi[ok, r, c], "o-", color=color, label=f"{name} ({ok.sum()} clear dates)")
axes[1].set(xlabel="Day of year", ylabel="NDVI", xlim=(0, 366), ylim=(-0.2, 1),
            title="Input time series (clear observations only)")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Part 5 · Computing the embeddings

Now the model runs on your computer. The image is processed in strips of rows so you can follow the progress. Because the model treats each pixel independently, splitting the image this way does not change the result.

Pixels without any clear observation get no embedding (`NaN`). The result is saved, so you can later reload it without recomputing.

⏱ About 1 to 3 minutes for the small study area on a laptop or Colab CPU, much less on a Colab GPU.

In [ ]:
EMB_FILE = OUT / f"embeddings_{AOI_NAME}_{YEAR}_{MODEL}.npy"
RECOMPUTE = False      # set True to force a new computation, e.g. after changing the data settings

if EMB_FILE.exists() and not RECOMPUTE:
    embeddings = np.load(EMB_FILE)
    print(f"Loaded saved embeddings from {EMB_FILE.name}")
else:
    embeddings = np.full((H, W, 128), np.nan, dtype=np.float32)
    ROWS_PER_STRIP = max(1, 20000 // W)          # roughly 20,000 pixels per strip
    t0 = time.time()
    with torch.inference_mode():                 # no training, so no gradients needed (faster, less memory)
        for r0 in range(0, H, ROWS_PER_STRIP):
            r1 = min(r0 + ROWS_PER_STRIP, H)
            embeddings[r0:r1] = encode_tile(model, s2_bands=s2[:, r0:r1], s2_doys=doys,
                                            s2_masks=masks[:, r0:r1],
                                            batch_pixels=BATCH_PIXELS, device=DEVICE)
            elapsed = time.time() - t0
            remaining = elapsed / r1 * (H - r1)
            print(f"\r  {r1/H:6.1%} done | {elapsed/60:4.1f} min elapsed | about {remaining/60:4.1f} min left",
                  end="", flush=True)
    embeddings[valid_counts == 0] = np.nan       # no observation, no embedding
    np.save(EMB_FILE, embeddings)
    print(f"\nFinished in {(time.time() - t0)/60:.1f} min, saved to {EMB_FILE.name}")

valid = np.isfinite(embeddings).all(axis=-1)
print("Embedding array:", embeddings.shape, "| pixels with embedding:", valid.sum())

### 5.1 What does an embedding look like?

Here are the 128 numbers of the two pixels you chose in Part 4.2. On their own, the individual numbers have **no physical meaning**: dimension 17 is not "greenness" or "moisture". What carries meaning is how vectors relate to each other. Pixels with similar surfaces and similar seasonal behaviour get similar vectors, which is what Part 6 makes use of.

Also note that the values of every pixel have mean 0 and standard deviation 1 (the final normalisation of the model), so the overall brightness of a pixel is not directly visible in its vector.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
for (name, (r, c)), color in zip(PIXELS.items(), ["red", "orange", "cyan", "magenta"]):
    if valid[r, c]:
        ax.plot(np.arange(128), embeddings[r, c], "-", color=color, lw=1.2, label=name)
for k in (16, 32, 64):
    ax.axvline(k - 0.5, color="grey", ls=":", lw=1)
    ax.text(k, ax.get_ylim()[1], f" first {k}", va="top", fontsize=8, color="grey")
ax.set(xlabel="Embedding dimension", ylabel="Value", title="Embedding vectors of the selected pixels", xlim=(-1, 128))
ax.legend(); plt.tight_layout(); plt.show()

### 5.2 Seeing 128 dimensions at once: PCA

A screen has three color channels, the embedding has 128 dimensions. **Principal component analysis (PCA)** finds the three directions along which the embeddings vary the most and maps them to red, green and blue. Pixels with similar colors have similar embeddings.

Keep in mind that the colors are arbitrary. Red does not mean anything in particular, and a re-run on another area gives a completely different color scheme. PCA also shows only part of the information, as the printed *explained variance* tells you.

In [ ]:
from sklearn.decomposition import PCA

X = embeddings[valid]                               # all valid pixels as a table: (pixels, 128)
pca = PCA(n_components=3, random_state=42)
pcs = pca.fit_transform(X)

pca_rgb = np.ones((H, W, 3), dtype=np.float32)      # white where no embedding exists
for c in range(3):
    lo, hi = np.percentile(pcs[:, c], [2, 98])     # robust stretch, ignores the 2 % extremes
    pca_rgb[valid, c] = np.clip((pcs[:, c] - lo) / (hi - lo), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(rgb_best); axes[0].set_title(f"True color, {dates[t_best]}")
axes[1].imshow(pca_rgb);  axes[1].set_title("TESSERA embeddings, first 3 principal components as RGB")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("Share of variance shown by the 3 components:", np.round(pca.explained_variance_ratio_, 3),
      f"→ {pca.explained_variance_ratio_.sum():.0%} in total")
plt.imsave(OUT / f"pca_rgb_{AOI_NAME}.png", pca_rgb)

**Check your understanding**

- Which landscape structures stand out in the PCA image that are hard to see in the true-color image, and vice versa?
- Two meadows look the same in the July image but have different colors in the PCA image. What could be the reason? (Hint: think about Part 4.2.)
- About how much of the total variation do the three components show? What does that mean for the interpretation of the image?

## Part 6 · Exploring the embedding space

All analyses in this part use the embeddings computed above, so they run in seconds and you can experiment freely.

**Important:** Unsupervised groups and similarities are **not land cover classes**. They show which pixels the model considers similar. Whether that corresponds to a meaningful class can only be checked with independent reference data.

### 6.1 K-Means clustering

K-Means sorts all pixels into `N_CLUSTERS` groups so that the embeddings within a group are as similar as possible. The algorithm is fitted on a random sample of pixels to keep it fast and then applied to all pixels. The cluster numbers are arbitrary labels.

**✏️ Your turn.** Try 4, 6 and 10 clusters. Which landscape units stay stable, which ones split up?

In [ ]:
from sklearn.cluster import KMeans
from matplotlib.colors import ListedColormap

N_CLUSTERS = 6
N_DIMS = 128           # how many embedding dimensions to use (16, 32, 64 or 128, see 6.3)
SAMPLE_SIZE = 50_000   # pixels used to fit K-Means

rng = np.random.default_rng(42)
sample = rng.choice(len(X), size=min(SAMPLE_SIZE, len(X)), replace=False)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit(X[sample, :N_DIMS])
labels = kmeans.predict(X[:, :N_DIMS])

cluster_map = np.full((H, W), np.nan)
cluster_map[valid] = labels
cmap_clusters = plt.get_cmap("tab10", N_CLUSTERS)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(rgb_best); axes[0].set_title("True color")
im = axes[1].imshow(cluster_map, cmap=cmap_clusters, vmin=-0.5, vmax=N_CLUSTERS - 0.5, interpolation="nearest")
axes[1].set_title(f"K-Means, {N_CLUSTERS} clusters ({N_DIMS} dimensions)")
for ax in axes: ax.axis("off")
fig.colorbar(im, ax=axes[1], ticks=range(N_CLUSTERS), fraction=0.046, pad=0.04, label="Cluster (arbitrary number)")
plt.tight_layout(); plt.show()
print("Pixels per cluster:", np.bincount(labels, minlength=N_CLUSTERS))

### 6.2 Similarity search: "find more pixels like this one"

Pick a reference pixel and compute how similar every other pixel's embedding is to it. We use **cosine similarity**, which compares the direction of two vectors. +1 means identical direction, 0 unrelated, −1 opposite. This is the basic operation behind embedding-based search tools: you point at one example and get all places that look and behave alike.

**✏️ Your turn.** Choose a reference pixel on a meadow, then on the river, then in a village. Use the row/column axes of the true-color image. How specific is the search in each case?

In [ ]:
REFERENCE = (H // 2, W // 2)    # (row, col) of the reference pixel

ref_row, ref_col = REFERENCE
if not valid[ref_row, ref_col]:
    raise ValueError("This pixel has no embedding (no clear observation). Choose another one.")
ref = embeddings[ref_row, ref_col]
similarity = (X @ ref) / (np.linalg.norm(X, axis=1) * np.linalg.norm(ref))

sim_map = np.full((H, W), np.nan)
sim_map[valid] = similarity

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(rgb_best); axes[0].set_title("True color with reference pixel (row/col axes)")
im = axes[1].imshow(sim_map, cmap="magma", vmin=0, vmax=1)
axes[1].set_title("Cosine similarity to the reference pixel"); axes[1].axis("off")
for ax in axes:
    ax.scatter(ref_col, ref_row, marker="x", s=120, c="cyan", linewidths=2.5)
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="Cosine similarity (values < 0 shown black)")
plt.tight_layout(); plt.show()
print(f"Pixels with similarity > 0.8: {(similarity > 0.8).mean():.1%} of the area")

### 6.3 How many dimensions are needed? (Matryoshka)

TESSERA's first 16, 32 or 64 dimensions are meant to work as a smaller embedding on their own. Smaller embeddings save storage and computing time, which matters at continental scale. But how much is lost?

The cell clusters the same pixels once with the first 16 and once with all 128 dimensions and compares the two maps with the **adjusted Rand index** (ARI). An ARI of 1 means identical groupings, 0 means no more agreement than chance.

**✏️ Your turn.** Repeat with `SMALL_DIMS = 32` and `64`. From which point on do the results barely change?

In [ ]:
from sklearn.metrics import adjusted_rand_score

SMALL_DIMS = 16

km_small = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit(X[sample, :SMALL_DIMS])
km_full = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit(X[sample])
labels_small, labels_full = km_small.predict(X[:, :SMALL_DIMS]), km_full.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, lab, n in zip(axes, [labels_small, labels_full], [SMALL_DIMS, 128]):
    m = np.full((H, W), np.nan); m[valid] = lab
    ax.imshow(m, cmap=cmap_clusters, vmin=-0.5, vmax=N_CLUSTERS - 0.5, interpolation="nearest")
    ax.set_title(f"K-Means with the first {n} dimensions"); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Adjusted Rand index ({SMALL_DIMS} vs. 128 dimensions): {adjusted_rand_score(labels_small, labels_full):.2f}")

### 6.4 UMAP: a map of the embedding space (optional)

UMAP projects the 128-dimensional vectors onto a 2D plane while trying to keep neighbours close together. Each dot is a pixel, colored by its K-Means cluster from 6.1. Tight, separate groups suggest that the model distinguishes these pixels clearly.

Be careful with the interpretation. UMAP axes have no meaning, distances between groups are not reliable, and the picture changes with the settings. Since the colors come from K-Means, a clean separation is also not independent proof that the clusters are "real".

In [ ]:
try:
    import umap
except ImportError:
    umap = None
    print("UMAP is not installed. Install it in the Anaconda Prompt with `python -m pip install umap-learn`, "
          "restart the kernel, or simply skip this cell.")

if umap is not None:
    UMAP_PIXELS, UMAP_NEIGHBORS = 5000, 15
    idx = rng.choice(len(X), size=min(UMAP_PIXELS, len(X)), replace=False)
    projection = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, min_dist=0.1, random_state=42).fit_transform(X[idx])

    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(projection[:, 0], projection[:, 1], c=labels[idx], cmap=cmap_clusters,
                    vmin=-0.5, vmax=N_CLUSTERS - 0.5, s=5, alpha=0.7)
    fig.colorbar(sc, ax=ax, ticks=range(N_CLUSTERS), label="K-Means cluster")
    ax.set(xlabel="UMAP 1", ylabel="UMAP 2", title=f"UMAP of {len(idx):,} random pixels")
    plt.tight_layout(); plt.show()

## Part 7 · Export to QGIS (optional)

The cell writes GeoTIFFs that you can open in QGIS together with other data: the first 16 embedding dimensions, the PCA image, the cluster map and the similarity map. Set `EXPORT_ALL_128 = True` to also export all 128 dimensions (about 0.5 KB per pixel, so roughly 200 MB for the small study area).

In [ ]:
import rasterio
from rasterio.transform import Affine

EXPORT_ALL_128 = False

def write_tif(path, array, nodata=-9999.0):
    "Write a (rows, cols) or (rows, cols, bands) array as GeoTIFF on the grid of the input data."
    data = array[..., None] if array.ndim == 2 else array
    data = np.nan_to_num(data.astype(np.float32), nan=nodata)
    profile = dict(driver="GTiff", height=H, width=W, count=data.shape[-1], dtype="float32",
                   crs=CRS_WKT, transform=Affine(*TRANSFORM), nodata=nodata, compress="deflate")
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(np.moveaxis(data, -1, 0))
    print("Saved", path)

write_tif(OUT / f"embeddings16_{AOI_NAME}.tif", embeddings[..., :16])
pca_export = np.where(valid[..., None], pca_rgb, np.nan)
write_tif(OUT / f"pca_rgb_{AOI_NAME}.tif", pca_export)
write_tif(OUT / f"kmeans{N_CLUSTERS}_{AOI_NAME}.tif", cluster_map)
write_tif(OUT / f"similarity_{AOI_NAME}.tif", sim_map)
if EXPORT_ALL_128:
    write_tif(OUT / f"embeddings128_{AOI_NAME}.tif", embeddings)

## Part 8 · Reflection

Answer the following questions in a few sentences each. They are about understanding, not about the code.

1. **Input matters.** In Part 4 we removed an offset of 1000 and masked clouds. What would happen to the embeddings if we had skipped these steps? Would you notice it just by looking at the PCA image?
2. **Time series vs. single image.** Find an example in your area where two surfaces look alike in the true-color image but end up in different clusters, or the other way round. Explain the difference with the NDVI time series from Part 4.2.
3. **Clusters are not classes.** Your K-Means map probably looks like a land cover map. What would you need to turn it into one, and how would you check its accuracy?
4. **Fair comparison.** Someone claims "embeddings are better than spectral indices". How would you design a test that could actually support or reject this claim? Think of a task, reference data, a baseline and how you split training and test data in space.
5. **Limitations.** Name two reasons why the embeddings in this exercise differ from the official TESSERA product, and one situation in which you would expect them to fail.

### Summary

- A pixel embedding is a compact vector that summarises a pixel's spectral and seasonal behaviour. Single dimensions have no physical meaning, relationships between vectors do.
- TESSERA derives embeddings from the time series of clear observations of each pixel, including the day of year of each observation.
- A pretrained model only works as intended if the input matches its training data: band order, scaling and offset, cloud mask.
- PCA, clustering, similarity search and UMAP are tools to *explore* embeddings. They generate hypotheses, not validated results.
- Whether embeddings are useful for a specific task has to be shown with reference data and a fair comparison against simpler baselines.

## Part 9 · Download your results (Colab)

In Colab, files are deleted when the session ends. This cell packs the figures and GeoTIFFs into one zip file and downloads it to your computer. The large raw data (`.npz`) and the full embedding array (`.npy`) are left out to keep the download small. If you work locally or saved to Google Drive, you do not need this cell.

In [ ]:
import zipfile

zip_path = OUT / f"tessera_results_{AOI_NAME}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.glob("*")):
        if f.suffix in (".png", ".tif"):
            zf.write(f, f.name)
print(f"Packed {len(zipfile.ZipFile(zip_path).namelist())} files into {zip_path.name}")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Running locally, the files are already in", OUT)